In [1]:
import requests
import pandas as pd

def safe_price(value):
    """تحويل السعر إلى رقم بصورة آمنة وتفادي أخطاء النصوص"""
    try:
        return float(value) / 100.0
    except (ValueError, TypeError):
        return 0.0

def fetch_story_games_by_genre(genre="Adventure"):
    url = f"https://steamspy.com/api.php?request=genre&genre={genre}"
    print(f"جاري سحب البيانات لفئة: '{genre}'...")
    
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'
    }
    
    try:
        response = requests.get(url, headers=headers, timeout=30)
        
        if response.status_code == 200:
            data = response.json()
            games_list = []
            
            for appid, game in data.items():
                games_list.append({
                    'appid': appid,
                    'name': game.get('name'),
                    'developer': game.get('developer'),
                    'publisher': game.get('publisher'),
                    'positive_reviews': game.get('positive', 0),
                    'negative_reviews': game.get('negative', 0),
                    'owners_range': game.get('owners', '0..0'),
                    'average_playtime_forever_min': game.get('average_forever', 0),
                    'median_playtime_forever_min': game.get('median_forever', 0),
                    'price_usd': safe_price(game.get('price')),
                    'initial_price_usd': safe_price(game.get('initialprice')),
                    'discount_pct': game.get('discount', 0)
                })
                
            df = pd.DataFrame(games_list)
            return df
        else:
            print(f"فشل في جلب البيانات. رمز الاستجابة: {response.status_code}")
            return None
            
    except Exception as e:
        print(f"حدث خطأ أثناء الاتصال: {e}")
        return None

if __name__ == "__main__":
    df_games = fetch_story_games_by_genre(genre="Adventure")
    
    if df_games is not None and not df_games.empty:
        print(f"\nتم سحب {len(df_games)} لعبة بنجاح!")
        print("\nمعاينة للبيانات المجلوبة:")
        print(df_games[['name', 'developer', 'price_usd', 'positive_reviews']].head())
        
        output_filename = "steam_story_games_raw.csv"
        df_games.to_csv(output_filename, index=False, encoding="utf-8-sig")
        print(f"\nتم حفظ البيانات بنجاح في ملف: '{output_filename}'")

جاري سحب البيانات لفئة: 'Adventure'...

تم سحب 34676 لعبة بنجاح!

معاينة للبيانات المجلوبة:
                  name         developer  price_usd  positive_reviews
0         Apex Legends           Respawn       0.00            668053
1  PUBG: BATTLEGROUNDS  PUBG Corporation       0.00           1520457
2             Palworld        Pocketpair      20.99            358266
3  New World: Aeternum      Amazon Games      59.99            196798
4   Black Myth: Wukong      Game Science      59.99           1111720

تم حفظ البيانات بنجاح في ملف: 'steam_story_games_raw.csv'


In [2]:
import pandas as pd
import numpy as np

# 1. قراءة البيانات الخام
df = pd.read_csv("steam_story_games_raw.csv")
print(f"إجمالي الألعاب الخام: {len(df)}")

# 2. تصفية الألعاب التي لا تملك تقييمات كافية (أقل من 10 تقييمات)
df['total_reviews'] = df['positive_reviews'] + df['negative_reviews']
df_clean = df[df['total_reviews'] >= 10].copy()

# 3. حساب نسبة التقييم الإيجابي (Review Ratio %)
df_clean['review_score_pct'] = round((df_clean['positive_reviews'] / df_clean['total_reviews']) * 100, 2)

# 4. تحويل وقت اللعب من دقائق إلى ساعات
df_clean['avg_playtime_hours'] = round(df_clean['average_playtime_forever_min'] / 60.0, 2)
df_clean['median_playtime_hours'] = round(df_clean['median_playtime_forever_min'] / 60.0, 2)

# 5. استخراج متوسط تقديري لعدد الملاك (Owners Midpoint)
def parse_owners(owners_str):
    try:
        if pd.isna(owners_str):
            return 0
        parts = str(owners_str).replace(',', '').split('..')
        if len(parts) == 2:
            min_o = float(parts[0].strip())
            max_o = float(parts[1].strip())
            return (min_o + max_o) / 2
        return 0
    except:
        return 0

df_clean['estimated_owners'] = df_clean['owners_range'].apply(parse_owners)

# 6. حساب تقدير الإيرادات المبيعة (Estimated Gross Revenue USD)
df_clean['estimated_revenue_usd'] = round(df_clean['estimated_owners'] * df_clean['price_usd'], 2)

# 7. تصنيف الألعاب حسب الفئة السعرية (Price Tier)
def categorize_price(price):
    if price == 0:
        return "Free to Play"
    elif price < 10:
        return "Budget (< $10)"
    elif price < 30:
        return "Mid-Tier ($10 - $30)"
    elif price < 60:
        return "High-Tier ($30 - $60)"
    else:
        return "Premium / AAA ($60+)"

df_clean['price_tier'] = df_clean['price_usd'].apply(categorize_price)

# 8. تصنيف تقريبي: Indie vs AAA/AA
def categorize_segment(row):
    if row['price_usd'] >= 49.99 or row['estimated_owners'] >= 2000000:
        return "AAA / Major Publisher"
    else:
        return "Indie / Mid-tier"

df_clean['game_segment'] = df_clean.apply(categorize_segment, axis=1)

# 9. تصنيف حالة التقييم (Rating Classification)
def categorize_rating(pct):
    if pct >= 90:
        return "Overwhelmingly Positive"
    elif pct >= 80:
        return "Very Positive"
    elif pct >= 70:
        return "Mostly Positive"
    elif pct >= 50:
        return "Mixed"
    else:
        return "Negative"

df_clean['rating_category'] = df_clean['review_score_pct'].apply(categorize_rating)

# 10. معاينة البيانات المنظفة وحفظها
print(f"\nإجمالي الألعاب بعد التنظيف والتصفية: {len(df_clean)}")
print("\nمعاينة للبيانات المنظفة:")
print(df_clean[['name', 'review_score_pct', 'rating_category', 'price_tier', 'game_segment', 'estimated_revenue_usd']].head())

# حفظ الملف المنظف
cleaned_filename = "steam_story_games_cleaned.csv"
df_clean.to_csv(cleaned_filename, index=False, encoding="utf-8-sig")
print(f"\n تم حفظ الملف المنظف بنجاح في: '{cleaned_filename}'")

إجمالي الألعاب الخام: 34676

إجمالي الألعاب بعد التنظيف والتصفية: 24695

معاينة للبيانات المنظفة:
                  name  review_score_pct          rating_category  \
0         Apex Legends             67.14                    Mixed   
1  PUBG: BATTLEGROUNDS             59.44                    Mixed   
2             Palworld             94.10  Overwhelmingly Positive   
3  New World: Aeternum             68.60                    Mixed   
4   Black Myth: Wukong             96.66  Overwhelmingly Positive   

              price_tier           game_segment  estimated_revenue_usd  
0           Free to Play  AAA / Major Publisher           0.000000e+00  
1           Free to Play  AAA / Major Publisher           0.000000e+00  
2   Mid-Tier ($10 - $30)  AAA / Major Publisher           1.574250e+09  
3  High-Tier ($30 - $60)  AAA / Major Publisher           4.499250e+09  
4  High-Tier ($30 - $60)  AAA / Major Publisher           4.499250e+09  

 تم حفظ الملف المنظف بنجاح في: 'steam_story_game

In [4]:
import pandas as pd

# 1. قراءة الملف المنظف
df = pd.read_csv("steam_story_games_cleaned.csv")

print("="*60)
print(" 1. ملخص عام للمشروع (Overall Project Summary)")
print("="*60)
print(f"إجمالي عدد الألعاب المحللة: {len(df):,}")
print(f"متوسط نسبة التقييم الإيجابي: {df['review_score_pct'].mean():.2f}%")
print(f"متوسط سعر اللعبة: ${df['price_usd'].mean():.2f}")
print(f"متوسط ساعات اللعب: {df['avg_playtime_hours'].mean():.1f} ساعة")

print("\n" + "="*60)
print(" 2. مقارنة Indie / Mid-tier مقابل AAA / Major Publishers")
print("="*60)
segment_summary = df.groupby('game_segment').agg(
    Game_Count=('appid', 'count'),
    Avg_Price_USD=('price_usd', 'mean'),
    Avg_Review_Score=('review_score_pct', 'mean'),
    Avg_Playtime_Hours=('avg_playtime_hours', 'mean'),
    Total_Est_Revenue_USD=('estimated_revenue_usd', 'sum')
).reset_index()

print(segment_summary.to_string(index=False))

print("\n" + "="*60)
print(" 3. توزيع التقييمات حسب الفئة السعرية (Price Tier Performance)")
print("="*60)
price_summary = df.groupby('price_tier').agg(
    Game_Count=('appid', 'count'),
    Avg_Review_Score=('review_score_pct', 'mean'),
    Avg_Playtime_Hours=('avg_playtime_hours', 'mean')
).reindex(["Free to Play", "Budget (< $10)", "Mid-Tier ($10 - $30)", "High-Tier ($30 - $60)", "Premium / AAA ($60+)"]).reset_index()

print(price_summary.to_string(index=False))

print("\n" + "="*60)
print(" 4. أعلى 5 ألعاب قصصية تحقيقاً للإيرادات التقديرية")
print("="*60)
top_revenue = df[['name', 'developer', 'price_usd', 'review_score_pct', 'estimated_revenue_usd']].sort_values(by='estimated_revenue_usd', ascending=False).head(5)
print(top_revenue.to_string(index=False))

print("\n" + "="*60)
print(" 5. أعلى 5 ألعاب قصصية من حيث ساعات اللعب (Engagement)")
print("="*60)
top_playtime = df[['name', 'developer', 'price_tier', 'avg_playtime_hours', 'review_score_pct']].sort_values(by='avg_playtime_hours', ascending=False).head(5)
print(top_playtime.to_string(index=False))

 1. ملخص عام للمشروع (Overall Project Summary)
إجمالي عدد الألعاب المحللة: 24,695
متوسط نسبة التقييم الإيجابي: 76.86%
متوسط سعر اللعبة: $8.49
متوسط ساعات اللعب: 0.0 ساعة

 2. مقارنة Indie / Mid-tier مقابل AAA / Major Publishers
         game_segment  Game_Count  Avg_Price_USD  Avg_Review_Score  Avg_Playtime_Hours  Total_Est_Revenue_USD
AAA / Major Publisher         449      45.108753         80.169777                 0.0           4.882823e+10
     Indie / Mid-tier       24246       7.807726         76.797170                 0.0           2.060137e+10

 3. توزيع التقييمات حسب الفئة السعرية (Price Tier Performance)
           price_tier  Game_Count  Avg_Review_Score  Avg_Playtime_Hours
         Free to Play        3057         73.107753                 0.0
       Budget (< $10)       15197         76.222600                 0.0
 Mid-Tier ($10 - $30)        5892         80.290528                 0.0
High-Tier ($30 - $60)         490         80.306551                 0.0
 Premium / AAA ($6